In [2]:
# --------------------------------- Part 1: Imports ---------------------------------
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import time
import pickle
import os

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, precision_recall_curve, auc, accuracy_score)
from scipy.stats import ttest_rel
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Dense, LSTM, Conv1D, Flatten, concatenate, Dropout,
                                     Multiply, Reshape, BatchNormalization, GlobalAveragePooling1D,
                                     Lambda)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [3]:
import os
# Save df_final as a .csv file
os.chdir(r'D:\sample_dataset')
df=pd.read_csv('df_final_cleaned.csv', low_memory=False)

In [4]:
df.shape

(1500000, 22)

In [5]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import mutual_info_classif

In [6]:
selected_features=[ 0 , 1,  2,  3,  4,  5,  8,  9, 10, 15, 16, 17, 18, 20]

In [7]:
from sklearn.preprocessing import LabelEncoder

# Apply Label Encoding for all object-type columns
label_encoders = {}
for column in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[column] = le.fit_transform(df[column].astype(str))
    label_encoders[column] = le


In [8]:
# Convert indices in selected_features to column names
selected_feature_names = df.columns[selected_features]

In [9]:
# Select the features and labels using the column names
X = df[selected_feature_names].values
y = df['Label'].values  

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [11]:
X.shape

(1500000, 14)

In [12]:
# Encode the labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_encoded = pd.get_dummies(y_encoded).values  # One-hot encode for multiclass classification


In [13]:
# Define CNN Model
def create_cnn_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = Conv1D(filters=64, kernel_size=3, activation='relu', padding='same')(inputs)
    x = BatchNormalization()(x)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)  # Output layer for multiclass classification
    model = Model(inputs, output)
    return model
# Define LSTM Model
def create_lstm_model(input_shape, num_classes):
    inputs = Input(shape=input_shape)
    x = LSTM(64, return_sequences=True)(inputs)
    x = BatchNormalization()(x)
    x = LSTM(32)(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)  # Output layer for multiclass classification
    model = Model(inputs, output)
    return model
# Define FNN Model
def create_fnn_model(input_shape, num_classes):
    inputs = Input(shape=(input_shape,))
    x = Dense(128, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)  # Output layer for multiclass classification
    model = Model(inputs, output)
    return model
# Define attention mechanism
def attention_mechanism(inputs):
    attention_weights = Dense(inputs.shape[-1], activation='softmax', name='attention_weights')(inputs)
    attention_output = Multiply(name='attention_output')([inputs, attention_weights])
    return attention_output, attention_weights



In [14]:
from memory_profiler import memory_usage

In [15]:
def train_ensemble():
    return ensemble_model.fit(
        [X_train_cnn, X_train_cnn, X_train], y_train,
        epochs=10,
        batch_size=128,
        validation_split=0.1,
        verbose=0,
        validation_data=([X_test_cnn, X_test_cnn, X_test], y_test)
    )

In [17]:

# Convert one-hot labels back to class labels for StratifiedKFold
y_labels = np.argmax(y_encoded, axis=1)
# --------------------------------- Part 4: 5-Fold Cross-Validation Setup ---------------------------------
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

ensemble_accs, cnn_accs, lstm_accs, fnn_accs = [], [], [], []
ensemble_times, cnn_times, lstm_times, fnn_times = [], [], [], []
ensemble_memory_usages, cnn_memory_usages, lstm_memory_usages, fnn_memory_usages = [], [], [], []
all_y_test = []
all_ensemble_pred = []
all_cnn_pred=[]
all_lstm_pred=[]
all_fnn_pred=[]
all_ensemble_prob = []
all_cm_ensemble = []
all_cm_cnn = []
all_cm_lstm = []
all_cm_fnn = []
attention_weights=[]
avg_atts=[]
fold = 1
for train_idx, test_idx in kfold.split(X, y_labels):
    print(f"\n=== Fold {fold} ===")
    fold += 1

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
    X_test_cnn = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)
    # Number of classes in the dataset
    num_classes = y_train.shape[1]
    # CNN
    cnn_model = create_cnn_model(X_train_cnn.shape[1:], num_classes)
    cnn_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    cnn_mem_usage, cnn_history = memory_usage(
    (cnn_model.fit, (X_train_cnn, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
    interval=0.1,
    retval=True )
    #cnn_history = cnn_model.fit(X_train_cnn, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    cnn_times.append(end - start)
    cnn_peak_memory = max(cnn_mem_usage)
    cnn_memory_usages.append(cnn_peak_memory)
    cnn_pred = (cnn_model.predict(X_test_cnn,verbose=0) > 0.5).astype(int)
    cnn_accs.append(accuracy_score(y_test, cnn_pred))
    
    # LSTM
  
    lstm_model = create_lstm_model(X_train_cnn.shape[1:], num_classes)
    lstm_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    lstm_mem_usage, lstm_history = memory_usage(
    (lstm_model.fit, (X_train_cnn, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
     interval=0.1,retval=True)
    #lstm_history= lstm_model.fit(X_train_cnn, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    lstm_times.append(end - start)
    lstm_peak_memory = max(lstm_mem_usage)
    lstm_memory_usages.append(lstm_peak_memory)
    lstm_pred = (lstm_model.predict(X_test_cnn,verbose=0) > 0.5).astype(int)
    lstm_accs.append(accuracy_score(y_test, lstm_pred))
    
     # FNN
    
    fnn_model = create_fnn_model(X_train.shape[1], num_classes)
    fnn_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    fnn_mem_usage, fnn_history = memory_usage(
    (fnn_model.fit, (X_train, y_train), {'epochs': 50, 'batch_size': 64, 'validation_split': 0.1, 'verbose': 0}),
    interval=0.1,
    retval=True)
    #fnn_history=fnn_model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.1, verbose=0)
    end = time.time()
    fnn_times.append(end - start)
    fnn_peak_memory = max(fnn_mem_usage)
    fnn_memory_usages.append(fnn_peak_memory)
    fnn_pred = (fnn_model.predict(X_test,verbose=0) > 0.5).astype(int)
    fnn_accs.append(accuracy_score(y_test, fnn_pred))
    
     # Ensemble
   
    # Get validation accuracy for dynamic weighting
    cnn_val_accuracy = cnn_model.evaluate(X_test_cnn, y_test, verbose=0)[1]
    lstm_val_accuracy = lstm_model.evaluate(X_test_cnn, y_test, verbose=0)[1]
    fnn_val_accuracy = fnn_model.evaluate(X_test, y_test, verbose=0)[1]
    
    # Calculate dynamic weights based on validation accuracy
    total_accuracy = cnn_val_accuracy + lstm_val_accuracy + fnn_val_accuracy
    weights = {
    'cnn': cnn_val_accuracy / total_accuracy,
    'lstm': lstm_val_accuracy / total_accuracy,
    'fnn': fnn_val_accuracy / total_accuracy
    }
    # Weighted outputs using Lambda layers
    cnn_weighted_output = Lambda(lambda x: x * weights['cnn'])(cnn_model.output)
    lstm_weighted_output = Lambda(lambda x: x * weights['lstm'])(lstm_model.output)
    fnn_weighted_output = Lambda(lambda x: x * weights['fnn'])(fnn_model.output)
    # Combine weighted outputs
    combined = concatenate([cnn_weighted_output, lstm_weighted_output, fnn_weighted_output], name='combined_features')
    # Apply attention mechanism on combined output
    attention_output, attention_tensor = attention_mechanism(combined)
    # Final output layer for multiclass classification
    output = Dense(num_classes, activation='softmax', name='output_layer')(attention_output)
    
    # Directly combine the model outputs without attention for testing
    combined = concatenate([cnn_weighted_output, lstm_weighted_output, fnn_weighted_output], name='combined_features')



   # Define the complete ensemble model
    ensemble_model = Model(inputs=[cnn_model.input, lstm_model.input, fnn_model.input], outputs=output)
    # This returns both the final classification output and attention weights
    ensemble_model_full = Model(inputs=[cnn_model.input, lstm_model.input, fnn_model.input],
                            outputs=[output, attention_tensor])

    # Compile and train the simplified model
    ensemble_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    start = time.time()
    ensemble_mem_usage, ensemble_history = memory_usage(
    train_ensemble,interval=0.1,retval=True)
    #ensemble_history=ensemble_model.fit(
     #   [X_train_cnn, X_train_cnn, X_train], y_train,
       # validation_data=([X_test_cnn, X_test_cnn, X_test], y_test),
        #epochs=50, batch_size=128, verbose=0)
    ensemble_peak_memory = max(ensemble_mem_usage)
    ensemble_memory_usages.append(ensemble_peak_memory)
    end = time.time()
    ensemble_times.append(end - start)
    ensemble_output, att_weights = ensemble_model_full.predict([X_test_cnn, X_test_cnn, X_test], verbose=0)
    ensemble_pred = np.argmax(ensemble_output, axis=1)
    y_true = np.argmax(y_test, axis=1)
    ensemble_accs.append(accuracy_score(y_true, ensemble_pred))

    all_y_test.append(y_test)
    all_ensemble_pred.append(ensemble_pred)
    all_ensemble_prob.append(ensemble_model.predict([X_test_cnn, X_test_cnn, X_test],verbose=0))  # Probabilities for ROC/PR
    ensemble_probs, att_weights = ensemble_model_full.predict([X_test_cnn, X_test_cnn, X_test], verbose=0)
    attention_weights.append(att_weights)  # Now att_weights is a real NumPy array
    avg_atts.append(np.mean(att_weights, axis=0))  # Mean over all test samples
    y_pred_ensemble = ensemble_model.predict([X_test_cnn, X_test_cnn, X_test],verbose=0)
    y_pred_classes_ensemble = np.argmax(y_pred_ensemble, axis=1)
    y_true_classes = np.argmax(y_test, axis=1)
    cm_ensemble = confusion_matrix(y_true_classes, y_pred_classes_ensemble)
    all_cm_ensemble.append(cm_ensemble)
    y_pred_classes_cnn = np.argmax(cnn_pred, axis=1)
    cm_cnn=confusion_matrix(y_true_classes,  y_pred_classes_cnn)
    all_cm_cnn.append(cm_cnn)
    y_pred_classes_lstm = np.argmax(lstm_pred, axis=1)
    cm_lstm=confusion_matrix(y_true_classes,y_pred_classes_lstm)
    all_cm_lstm.append(cm_lstm)
    y_pred_classes_fnn = np.argmax(fnn_pred, axis=1)
    cm_fnn=confusion_matrix(y_true_classes,  y_pred_classes_fnn)
    all_cm_fnn.append(cm_fnn)
    


=== Fold 1 ===


=== Fold 2 ===

=== Fold 3 ===

=== Fold 4 ===

=== Fold 5 ===


In [37]:
attack_counts = df['Label'].value_counts()
print(attack_counts)

0    1372539
2      62383
3      17502
8      17159
5       8980
1       8046
6       7458
9       3762
7       1663
4        508
Name: Label, dtype: int64


In [31]:

# --------------------------------- Part 5: Report Results ---------------------------------
def report_scores(name, scores, times, memories):
    print(f"{name}: Accuracy = {np.mean(scores):.4f} ± {np.std(scores):.4f}, "
          f"Time = {np.mean(times):.2f}s ± {np.std(times):.2f}s, "
          f"Memory = {np.mean(memories):.2f} MiB ± {np.std(memories):.2f} MiB")
print("\n=== 5-Fold Cross-validation Results ===")
report_scores("CNN", cnn_accs, cnn_times, cnn_memory_usages)
report_scores("LSTM", lstm_accs, lstm_times, lstm_memory_usages)
report_scores("FNN", fnn_accs, fnn_times, fnn_memory_usages)
report_scores("Ensemble", ensemble_accs, ensemble_times, ensemble_memory_usages)


=== 5-Fold Cross-validation Results ===
CNN: Accuracy = 0.8571 ± 0.0049, Time = 5053.29s ± 659.70s, Memory = 1176.10 MiB ± 234.68 MiB
LSTM: Accuracy = 0.8743 ± 0.0062, Time = 17499.96s ± 1434.59s, Memory = 1242.57 MiB ± 226.55 MiB
FNN: Accuracy = 0.8752 ± 0.0123, Time = 3665.14s ± 79.65s, Memory = 1254.17 MiB ± 219.44 MiB
Ensemble: Accuracy = 0.9034 ± 0.0002, Time = 4377.13s ± 144.94s, Memory = 1309.69 MiB ± 176.29 MiB


In [32]:

# --------------------------------- Part 6: Statistical Significance Testing ---------------------------------
print("\n=== Paired t-tests ===")
print("Ensemble vs CNN:", ttest_rel(ensemble_accs, cnn_accs))
print("Ensemble vs LSTM:", ttest_rel(ensemble_accs, lstm_accs))
print("Ensemble vs FNN:", ttest_rel(ensemble_accs, fnn_accs))


=== Paired t-tests ===
Ensemble vs CNN: TtestResult(statistic=18.280272301500137, pvalue=5.267514397984495e-05, df=4)
Ensemble vs LSTM: TtestResult(statistic=9.37491048218807, pvalue=0.0007211702136799581, df=4)
Ensemble vs FNN: TtestResult(statistic=4.666738391368625, pvalue=0.009541483301216087, df=4)


In [33]:
class_names =  [
    'Normal','fuzzing', 'http-flood', 'http-loris', 'http-smuggle', 'http2-concurrent','http2-pause','quic-enc','quic-flood', 
      'quic-loris'
]

In [38]:

report = classification_report(y_true_classes , y_pred_classes_ensemble, target_names=class_names)
print("Ablation-without WGANGP - Ensemble Model (5-Fold CV):H23Q-Multiclass Classification")
print(report)

Ablation-without WGANGP - Ensemble Model (5-Fold CV):H23Q-Multiclass Classification
                  precision    recall  f1-score   support

          Normal       1.00      0.90      0.95   1372539
         fuzzing       0.75      0.88      0.81      8046
      http-flood       0.85      0.92      0.88     62383
      http-loris       0.61      0.82      0.70     17502
    http-smuggle       0.03      0.98      0.05       508
http2-concurrent       0.08      0.51      0.13      8980
     http2-pause       0.03      0.20      0.05      7458
        quic-enc       0.36      0.90      0.52      1663
      quic-flood       0.83      0.88      0.85     17159
      quic-loris       0.83      0.69      0.75      3762

        accuracy                           0.90   1500000
       macro avg       0.54      0.77      0.57   1500000
    weighted avg       0.97      0.90      0.93   1500000

